# Advanced 2 — XGBoost + Optuna (байєсівський тюнінг)

Додаємо градієнтний бустинг **XGBoost** і підбираємо гіперпараметри **Optuna** (мінімізуємо OOF-Brier на train — leakage-clean, узгоджено з критерієм вибору моделі проєкту). Порівнюємо head-to-head проти простої LogReg.

> Залежності: `%run 03_data_prep.ipynb`, `%run 04_evaluate.ipynb`.

In [ ]:
%run 03_data_prep.ipynb
%run 04_evaluate.ipynb

### Тюнінг XGBoost через Optuna (30 trials, OOF-Brier)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, optuna, xgboost as xgb
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import brier_score_loss
from sklearn.calibration import CalibratedClassifierCV

Xtr, ytr, Xte, yte = b['X_train'], b['y_train'], b['X_test'], b['y_test']
cv = StratifiedKFold(5, shuffle=True, random_state=42)
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = dict(
        max_depth=trial.suggest_int('max_depth', 2, 5),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        n_estimators=trial.suggest_int('n_estimators', 50, 400),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 30))
    m = xgb.XGBClassifier(**params, eval_metric='logloss', tree_method='hist',
                          random_state=42, n_jobs=-1)
    oof = cross_val_predict(m, Xtr, ytr, cv=cv, method='predict_proba')[:, 1]
    return brier_score_loss(ytr, oof)   # мінімізуємо OOF-Brier

study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30, show_progress_bar=False)
print('best OOF-Brier:', round(study.best_value, 4))
print('best params:', study.best_params)

### Калібрування найкращого XGB і порівняння з LogReg

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

best = xgb.XGBClassifier(**study.best_params, eval_metric='logloss', tree_method='hist',
                         random_state=42, n_jobs=-1)
xgb_cal = CalibratedClassifierCV(best, method='sigmoid', cv=5).fit(Xtr, ytr)
p_xgb = xgb_cal.predict_proba(Xte)[:, 1]

lr = Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
               ('clf',LogisticRegression(max_iter=2000,C=0.5,random_state=42))]).fit(Xtr,ytr)
p_lr = lr.predict_proba(Xte)[:, 1]

import pandas as pd
res = {'logreg': evaluate.binary_metrics(yte, p_lr, 0.5),
       'xgb_tuned': evaluate.binary_metrics(yte, p_xgb, 0.5)}
print(pd.DataFrame(res).T[['roc_auc','pr_auc','brier','accuracy']].round(3))
print('XGB test AUC 95% CI:', [round(x,3) for x in evaluate.bootstrap_ci(yte, p_xgb, evaluate._safe_auc)])

### Висновок

Очікувано на цих даних тюнінгований XGBoost **не б'є** просту калібровану LogReg (дрейф + крихітний датасет; бустинг легко перекалібровується). Це підтверджує рішення проєкту деплоїти простішу модель — і показує, що ми це **перевірили**, а не припустили. Якщо ж XGB кращий за Brier — онови `DEPLOY_CANDIDATES` у `00_config`.